# Whisper 数据准备与依赖安装

在远程 GPU 服务器上运行此 notebook，准备 Whisper 微调所需的数据和环境。

## 1. 安装依赖

In [ ]:
import subprocess
import sys

print("安装 peft, librosa...")
subprocess.run([sys.executable, "-m", "pip", "install", "peft", "librosa", "-q"])
print("安装完成!")

## 2. 检查现有数据

In [ ]:
import os
import json

# 检查可用的数据
paths = {
    "SenseVoice训练集": "/mnt/data/prepared_data/sensevoice/train.jsonl",
    "SenseVoice验证集": "/mnt/data/prepared_data/sensevoice/val.jsonl",
    "Nano训练集": "/mnt/data/prepared_data/funasr_nano/train.jsonl",
    "Nano验证集": "/mnt/data/prepared_data/funasr_nano/val.jsonl",
}

for name, path in paths.items():
    if os.path.exists(path):
        count = sum(1 for _ in open(path))
        print(f"  {name}: {count} 条 OK")
    else:
        print(f"  {name}: 不存在")

## 3. 准备 Whisper 数据

将现有 SenseVoice 格式数据转换为 Whisper 格式（只取部分样本测试）。

In [ ]:
import os
import json

WHISPER_DATA_DIR = "/mnt/data/prepared_data/whisper"
os.makedirs(WHISPER_DATA_DIR, exist_ok=True)

# 使用 SenseVoice 数据作为基础
SENSEVOICE_TRAIN = "/mnt/data/prepared_data/sensevoice/train.jsonl"
SENSEVOICE_VAL = "/mnt/data/prepared_data/sensevoice/val.jsonl"

MAX_TRAIN = 500   # 取 500 条训练样本
MAX_VAL = 50     # 取 50 条验证样本

def convert_to_whisper_format(input_path, output_path, max_items=None):
    """将 SenseVoice 格式转换为 Whisper JSONL 格式"""
    count = 0
    with open(input_path, 'r', encoding='utf-8') as fin, \
         open(output_path, 'w', encoding='utf-8') as fout:
        for line in fin:
            item = json.loads(line)
            # 转换字段名
            whisper_item = {
                'audio_filepath': item.get('source') or item.get('audio_filepath'),
                'text': item.get('target') or item.get('text'),
            }
            fout.write(json.dumps(whisper_item, ensure_ascii=False) + '\n')
            count += 1
            if max_items and count >= max_items:
                break
    return count

if os.path.exists(SENSEVOICE_TRAIN):
    train_count = convert_to_whisper_format(
        SENSEVOICE_TRAIN,
        os.path.join(WHISPER_DATA_DIR, 'train.jsonl'),
        MAX_TRAIN
    )
    print(f"训练集: {train_count} 条 -> {WHISPER_DATA_DIR}/train.jsonl")
else:
    print("SenseVoice 训练集不存在!")

if os.path.exists(SENSEVOICE_VAL):
    val_count = convert_to_whisper_format(
        SENSEVOICE_VAL,
        os.path.join(WHISPER_DATA_DIR, 'val.jsonl'),
        MAX_VAL
    )
    print(f"验证集: {val_count} 条 -> {WHISPER_DATA_DIR}/val.jsonl")
else:
    print("SenseVoice 验证集不存在!")

## 4. 验证数据

In [ ]:
import librosa

TRAIN_JSONL = "/mnt/data/prepared_data/whisper/train.jsonl"
VAL_JSONL = "/mnt/data/prepared_data/whisper/val.jsonl"

for path, label in [(TRAIN_JSONL, '训练集'), (VAL_JSONL, '验证集')]:
    if os.path.exists(path):
        count = sum(1 for _ in open(path))
        print(f"  {label}: {count} 条")
    else:
        print(f"  {label}: 不存在! {path}")

# 测试音频加载
if os.path.exists(TRAIN_JSONL):
    with open(TRAIN_JSONL) as f:
        sample = json.loads(f.readline())
    audio = sample.get('audio_filepath') or sample.get('source')
    text = sample.get('text')
    print(f"\n数据样例:")
    print(f"  音频: {audio}")
    print(f"  文本: {text}")
    
    if os.path.exists(audio):
        array, sr = librosa.load(audio, sr=16000)
        print(f"  音频长度: {len(array)/sr:.2f}s")
        print(f"  音频OK!")
    else:
        print(f"  音频文件不存在: {audio}")

## 5. 验证依赖

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

import transformers
print(f"transformers: {transformers.__version__}")

import peft
print(f"peft: {peft.__version__}")

import librosa
print(f"librosa: {librosa.__version__}")

import datasets
print(f"datasets: {datasets.__version__}")

print("\n所有依赖就绪!")

## 下一步

现在可以运行 `05_finetune_whisper_test.ipynb` 进行微调测试。